# Predikcija cijene polovnih automobila
Autor: **Anel Rakaj**

## Exploratory data analysis (EDA)


### 1. Load the Dataset
Set up the file path using `pathlib` and load the cars dataset into a pandas DataFrame

In [21]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/cars.csv")
df = pd.read_csv(DATA_PATH)

### 2. Initial Data Inspection
Preview the first few rows and a random sample of 5 records to understand the dataset's structure and contents.

In [22]:
df.head()
df.sample(5, random_state=42)

,make,model,priceUSD,year,condition,mileage(kilometers),fuel_type,volume(cm3),color,transmission,drive_unit,segment
19270,mitsubishi,carisma,2050,1999,with mileage,250000.0,petrol,1800.0,blue,mechanics,front-wheel drive,M
25927,ford,fusion,4500,2006,with mileage,160000.0,petrol,1400.0,burgundy,mechanics,front-wheel drive,M
23388,mercedes-benz,e-klass,3500,1999,with mileage,485000.0,diesel,2900.0,blue,mechanics,rear drive,E
53189,bmw,x3,21000,2013,with mileage,159000.0,diesel,2000.0,black,auto,NaN,J
34058,renault,megane,3990,2002,with mileage,331700.0,diesel,1900.0,other,mechanics,front-wheel drive,C


### 2.1 Check Dataset Dimensions  
Display the total number of rows and columns to verify the shape of the dataset

In [23]:
df.shape
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Rows: 56244
Columns: 12


### 3. List Feature Names
Extract and display all column names as a list to see the available attributes in the dataset.

In [24]:
df.columns.tolist()

['make',
 'model',
 'priceUSD',
 'year',
 'condition',
 'mileage(kilometers)',
 'fuel_type',
 'volume(cm3)',
 'color',
 'transmission',
 'drive_unit',
 'segment']

### 3.1 View Data Summary
Display the structural summary of the DataFrame, including data types, non-null counts, and memory usage.

In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 56244 entries, 0 to 56243
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   make                 56244 non-null  str    
 1   model                56244 non-null  str    
 2   priceUSD             56244 non-null  int64  
 3   year                 56244 non-null  int64  
 4   condition            56244 non-null  str    
 5   mileage(kilometers)  56244 non-null  float64
 6   fuel_type            56244 non-null  str    
 7   volume(cm3)          56197 non-null  float64
 8   color                56244 non-null  str    
 9   transmission         56244 non-null  str    
 10  drive_unit           54339 non-null  str    
 11  segment              50953 non-null  str    
dtypes: float64(2), int64(2), str(8)
memory usage: 5.1 MB


### 4.  Inspect the Target Variable
Preview the first few entries of `priceUSD`, which serves as the target variable for this analysis.

In [26]:
df["priceUSD"].head()

0    5500
1    5350
2    7000
3    3300
4    5200
Name: priceUSD, dtype: int64

### 4.1 Check Target Variable Data Type
Verify the data type of the `priceUSD` column to ensure it is correctly formatted for numerical analysis

In [27]:
df["priceUSD"].dtype

dtype('int64')

### 4.2 Inspect Unique Target Values
Preview the first 10 unique values of `priceUSD` to detect potential anomalies, formatting errors, or distinct price categories.

In [28]:
df["priceUSD"].unique()[:10]

array([5500, 5350, 7000, 3300, 5200, 3400, 5000, 7300, 6400, 6132])

## 5.  Identify Missing Values
Calculate the total number of missing entries per column and filter for features that contain missing data

In [29]:
missing_values = df.isna().sum()
missing_values[missing_values > 0]

volume(cm3)      47
drive_unit     1905
segment        5291
dtype: int64

### 5.1 Detect Pseudo-Missing Values
Scan object-type columns for string placeholders that mimic missing data (e.g., "NaN", "null", or empty strings) and count their occurrences.

In [30]:
missing_like_values = ["NaN", "nan", "NULL", "null", "None", "none", "", " "]

for column in df.columns:
   if df[column].dtype == "object":
       count = df[column].astype(str).str.strip().isin(missing_like_values).sum()
        
       if count > 0:
           print(f"{column}: {count}")

### 6. Identify Numerical Features
Filter and inspect the data types of columns to distinguish numerical variables from categorical ones for further analysis.

In [31]:
df.dtypes

make                       str
model                      str
priceUSD                 int64
year                     int64
condition                  str
mileage(kilometers)    float64
fuel_type                  str
volume(cm3)            float64
color                      str
transmission               str
drive_unit                 str
segment                    str
dtype: object

In [32]:
numeric_columns = ['priceUSD', 'year', 'mileage(kilometers)', 'volume(cm3)']

In [33]:
df[numeric_columns].dtypes

df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
priceUSD,56244.0,7415.456440,8316.959261,48.0,2350.0,5350.0,9807.5,235235.0
year,56244.0,2003.454840,8.144247,1910.0,1998.0,2004.0,2010.0,2019.0
mileage(kilometers),56244.0,244395.631020,321030.668382,0.0,137000.0,228500.0,310000.0,9999999.0
volume(cm3),56197.0,2104.860615,959.201633,500.0,1600.0,1996.0,2300.0,20000.0


### 6.1 Define Categorical Features
Identify and group the categorical columns representing specific car attributes for encoding and discrete data analysis.


In [34]:
categorical_columns = ['make', 'model', 'condition', 'fuel_type', 'transmission', 'drive_unit', 'segment']

In [35]:
for column in categorical_columns:
   print(column)
   print(df[column].nunique())
   print("-" * 40)

make
96
----------------------------------------
model
1034
----------------------------------------
condition
3
----------------------------------------
fuel_type
3
----------------------------------------
transmission
2
----------------------------------------
drive_unit
4
----------------------------------------
segment
9
----------------------------------------


In [36]:
for column in categorical_columns:
   print(df[column].value_counts(dropna=False))
   print("-" * 40)

make
volkswagen    6861
audi          4030
bmw           4013
opel          3779
renault       3713
              ... 
trabant          1
jac              1
asia             1
tagaz            1
saipa            1
Name: count, Length: 96, dtype: int64
----------------------------------------
model
passat      2086
5-seriya    1476
a6          1276
golf        1070
astra       1013
            ... 
xb             1
xc40           1
xjs            1
xt5            1
z3             1
Name: count, Length: 1034, dtype: int64
----------------------------------------
condition
with mileage    55278
with damage       512
for parts         454
Name: count, dtype: int64
----------------------------------------
fuel_type
petrol        36405
diesel        19792
electrocar       47
Name: count, dtype: int64
----------------------------------------
transmission
mechanics    36056
auto         20188
Name: count, dtype: int64
----------------------------------------
drive_unit
front-wheel drive       